# DarkNav - Model Architecture and Training

**Author:** Miriam Garcia Sollo  
**Date:** June 2026

This notebook covers the U-Net implementation and training of the three conditions:
- Condition A: scratch baseline
- Condition B: ImageNet pretrained encoder
- Condition C: NFW synthetic pretrain + fine-tune

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import torch
import json
from pathlib import Path
from torch.utils.data import DataLoader

from src.model import DarkNavUNet, verify_model_dimensions
from src.train import train, build_dataloaders, compute_metrics
from src.synthetic import SyntheticCraterDataset, RealCraterDataset

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

DATA_DIR = Path("../data")
PROC_DIR = DATA_DIR / "processed"
RUNS_DIR = Path("../runs")
DOCS_DIR = Path("../docs")
RUNS_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## 1. Architecture verification

In [ ]:
print("=== Condition A: scratch ===")
info_A = verify_model_dimensions(mode="scratch", verbose=True)
print()
print("=== Condition B: imagenet ===")
info_B = verify_model_dimensions(mode="imagenet", verbose=True)
print()
print(f"Parameter comparison:")
print(f"  A (scratch)  : {info_A['parameters']:>10,} params  {info_A['ram_mb']:.0f} MB")
print(f"  B (imagenet) : {info_B['parameters']:>10,} params  {info_B['ram_mb']:.0f} MB")

## 2. Forward pass and loss test

In [ ]:
from src.synthetic import make_single_crater

rng = np.random.default_rng(0)
batch_imgs, batch_masks = [], []
for _ in range(4):
    d, m = make_single_crater(r_s=rng.uniform(10, 25), noise_std=0.04, rng=rng)
    batch_imgs.append(d[np.newaxis])
    batch_masks.append(m[np.newaxis].astype(np.float32))

images = torch.from_numpy(np.stack(batch_imgs))
masks  = torch.from_numpy(np.stack(batch_masks))

model_test = DarkNavUNet(mode="scratch")
criterion  = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([9.0]))

logits = model_test(images)
loss   = criterion(logits, masks)
metrics = compute_metrics(logits.detach(), masks)

print(f"Input : {tuple(images.shape)}")
print(f"Output: {tuple(logits.shape)}")
print(f"Loss  : {loss.item():.4f}")
print(f"IoU   : {metrics['iou']:.4f}  (near 0 before training, expected)")

loss.backward()
print("Backward pass: OK")

## 3. Build DataLoaders

In [ ]:
pos_weight_path = DATA_DIR / "pos_weight.npy"
pos_weight = float(np.load(pos_weight_path)[0]) if pos_weight_path.exists() else 9.0
print(f"pos_weight: {pos_weight:.2f}")

synth_ds     = SyntheticCraterDataset(n_samples=1500, seed=42, augment=True)
synth_loader = DataLoader(synth_ds, batch_size=8, shuffle=True, num_workers=0)

train_real = PROC_DIR / "train_images.npy"
if train_real.exists():
    train_loader, val_loader, test_loader, _ = build_dataloaders(
        PROC_DIR, batch_size=8, use_synthetic_train=False
    )
    REAL_DATA = True
    print("Real data loaded.")
else:
    synth_val    = SyntheticCraterDataset(n_samples=100, seed=99, augment=False)
    train_loader = synth_loader
    val_loader   = DataLoader(synth_val, batch_size=8, shuffle=False, num_workers=0)
    test_loader  = val_loader
    REAL_DATA    = False
    print("Real data not found. Using synthetic for demo.")

## 4. Training

Set  for a quick run in the notebook.  
For the full experiment use the CLI:


In [ ]:
EPOCHS_DEMO = 5  # increase to 50 for full experiment

print("--- Condition A: scratch ---")
model_A  = DarkNavUNet(mode="scratch").to(DEVICE)
history_A = train(
    model=model_A, train_loader=train_loader, val_loader=val_loader,
    n_epochs=EPOCHS_DEMO, lr=1e-3, pos_weight=pos_weight,
    out_dir=RUNS_DIR / "condition_A", phase_name="condition_A",
    device=DEVICE, patience=10,
)

In [ ]:
print("--- Condition B: imagenet ---")
model_B  = DarkNavUNet(mode="imagenet").to(DEVICE)
history_B = train(
    model=model_B, train_loader=train_loader, val_loader=val_loader,
    n_epochs=EPOCHS_DEMO, lr=1e-4, pos_weight=pos_weight,
    out_dir=RUNS_DIR / "condition_B", phase_name="condition_B",
    device=DEVICE, patience=10,
)

In [ ]:
print("--- Condition C phase 1: NFW pretrain ---")
model_C = DarkNavUNet(mode="scratch").to(DEVICE)
history_C_pre = train(
    model=model_C, train_loader=synth_loader, val_loader=val_loader,
    n_epochs=EPOCHS_DEMO, lr=1e-3, pos_weight=pos_weight,
    out_dir=RUNS_DIR / "condition_C", phase_name="nfw_pretrain",
    device=DEVICE, patience=15,
)

print()
print("--- Condition C phase 2: fine-tune ---")
nfw_ckpt = RUNS_DIR / "condition_C" / "nfw_pretrain_best.pth"
if nfw_ckpt.exists():
    ckpt = torch.load(nfw_ckpt, map_location=DEVICE, weights_only=True)
    model_C.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded NFW checkpoint: val_IoU={ckpt['val_iou']:.4f}")

history_C_ft = train(
    model=model_C, train_loader=train_loader, val_loader=val_loader,
    n_epochs=EPOCHS_DEMO, lr=1e-4, pos_weight=pos_weight,
    out_dir=RUNS_DIR / "condition_C", phase_name="condition_C",
    device=DEVICE, patience=10,
)

## 5. Training curves

In [ ]:
def load_history(path):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return {}

h_A    = load_history(RUNS_DIR / "condition_A" / "condition_A_history.json")
h_B    = load_history(RUNS_DIR / "condition_B" / "condition_B_history.json")
h_C    = load_history(RUNS_DIR / "condition_C" / "condition_C_history.json")
h_Cpre = load_history(RUNS_DIR / "condition_C" / "nfw_pretrain_history.json")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, key, title in [
    (axes[0], "val_iou",  "Validation IoU per epoch"),
    (axes[1], "val_loss", "Validation Loss per epoch"),
]:
    if h_A.get(key):    ax.plot(h_A[key],    label="A: scratch",        color="#4878CF", lw=2)
    if h_B.get(key):    ax.plot(h_B[key],    label="B: imagenet",       color="#E87E5D", lw=2)
    if h_C.get(key):    ax.plot(h_C[key],    label="C: DarkNav",        color="#2A9D8F", lw=2)
    if h_Cpre.get(key): ax.plot(h_Cpre[key], label="C: NFW pretrain",  color="#2A9D8F", lw=1.2, ls="--", alpha=0.6)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(key.split("_")[1].upper())
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle("Training curves: all conditions", fontsize=12)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig19_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig19_training_curves.png")